# Price-Dividend Ratio Decomposition: Cash Flows vs. Discount Rates

**UCLA MFE — Empirical Methods, coursework (group assignment, 6 members)**

From the Campbell-Shiller present-value identity, the log price-dividend ratio decomposes as
$$pd_t = \text{constant} + \sum_{j=1}^\infty \rho^{j-1} E_t(\Delta d_{t+j}) - \sum_{j=1}^\infty \rho^{j-1} E_t(r_{t+j}).$$

The question this assignment answers: when the price-dividend ratio moves, is that because
expected future *cash flows* (dividend growth) changed, or because the *discount rate*
(expected return) changed? This matters for risk management because it separates two very
different sources of valuation risk — a cash-flow shock is about fundamentals, a discount-rate
shock is about time-varying risk premia / risk aversion.

## Part 1: Deriving the cash-flow and discount-rate components

Given a dividend-growth process $\Delta d_t = \mu + \varepsilon_{d,t}$ and a discount-rate
process driven by AR(1) risk aversion $\gamma_t = \bar\gamma + \phi(\gamma_{t-1}-\bar\gamma)+\varepsilon_{\gamma,t}$
with expected return $E_t[r_{t+1}] = r_f + \gamma_t \sigma_d^2$, the two components of $pd_t$
work out to:

$$CF_t = \frac{\mu}{1-\rho}, \qquad
DR_t = \frac{r_f + \bar\gamma\sigma_d^2}{1-\rho} + \frac{\sigma_d^2}{1-\rho\phi}(\gamma_t - \bar\gamma)$$

**Key result:** $CF_t$ is *constant* (dividend growth has constant conditional mean $\mu$, so
its discounted sum doesn't move with new information), while $DR_t$ is the only
time-varying piece, moving one-for-one with the risk-aversion shock $(\gamma_t - \bar\gamma)$.

## Part 2: Variance decomposition

Since $CF_t$ is constant, $\mathrm{Var}(CF_t) = 0$ and $\mathrm{Cov}(CF_t, DR_t) = 0$, so
$$pd_t = \text{constant} + CF_t - DR_t \;\Rightarrow\; \mathrm{Var}(pd_t) = \mathrm{Var}(DR_t)
\;\Rightarrow\; \frac{\mathrm{Var}(DR_t)}{\mathrm{Var}(pd_t)} = 1.$$

In this stylized model, **100% of price-dividend-ratio variance comes from the discount
rate** — by construction, since dividend growth here has no persistent/predictable component
to generate cash-flow-driven variation. Numerically, plugging in
$\mu=2\%, \phi=0.9, \sigma_\gamma=1.5, \bar\gamma=5, \sigma_d=12\%, \rho=0.96$:

In [ ]:
sigma_d2 = 0.12 ** 2          # 0.0144
sigma_d4 = sigma_d2 ** 2      # 0.00020736
sigma_gamma2 = 1.5 ** 2       # 2.25

phi, rho = 0.9, 0.96

term1 = 1 / (1 - phi**2)                       # 5.2632
term2 = rho**2 / (1 - rho*phi)**2              # 49.837

dr_shock_component = sigma_d4 * sigma_gamma2 * (term1 + term2)
var_returns = sigma_d2 + dr_shock_component

share_from_discount_rate_shocks = dr_shock_component / var_returns

print(f"sigma_d^2 = {sigma_d2:.4f}")
print(f"discount-rate-shock component of Var(r_t) = {dr_shock_component:.4f}")
print(f"Var(r_t) total = {var_returns:.4f}")
print(f"Share of stock-return variance from discount-rate shocks = {share_from_discount_rate_shocks:.4%}")

**Result:** with these parameters, discount-rate (future expected-return) shocks account
for about **64.1%** of the *variance of stock returns themselves* (a different, related
question from the pd-ratio decomposition above — this asks how much of realized return
variance, not pd-ratio variance, traces back to discount-rate news vs. cash-flow news). The
majority of short-run stock return volatility here comes from revisions to expected future
returns, not from news about dividends — consistent with the broader empirical finance
finding (Campbell-Shiller, Cochrane) that discount-rate variation dominates cash-flow variation
in explaining aggregate market volatility.

## Part 3: Simulating the model and checking predictability

Simulate 10,000 periods of dividend growth and risk aversion as AR(1) processes, construct the
implied pd-ratio and return series, and check the qualitative predictions above hold in
simulation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.api import VAR

# Simulate AR(1)
def simulate_ar1(T, phi, sigma, x0=0.0, rng=None):
    """
    x_t = phi * x_{t-1} + eps_t,  eps_t ~ N(0, sigma^2)
    Returns length T+1 array including x0 as the first element.
    """
    rng = np.random.default_rng() if rng is None else rng
    x = np.empty(T + 1)
    x[0] = x0
    eps = rng.normal(0.0, sigma, size=T)
    for t in range(1, T + 1):
        x[t] = phi * x[t - 1] + eps[t - 1]
    return x


def compute_pd_and_returns(delta_d, gamma, params):
    """
    Given simulated dividend growth delta_d and risk aversion gamma_t, compute
      - pd_t  (price-dividend ratio, demeaned)
      - r_t   (market return, demeaned)
    as linear functions of the state, per the Part 1/2 derivation:
      pd_t = b_pd_d * delta_d + b_pd_g * gamma_t
      r_t  = b_r_d  * delta_d + b_r_g  * gamma_t
    """
    b_pd_d = params.get("b_pd_d", 0.0)
    b_pd_g = params.get("b_pd_g", -1.0)   # higher gamma -> lower pd
    b_r_d = params.get("b_r_d", 1.0)
    b_r_g = params.get("b_r_g", 0.5)

    pd = b_pd_d * delta_d + b_pd_g * gamma
    r = b_r_d * delta_d + b_r_g * gamma
    return pd, r


T = 10_000
rng = np.random.default_rng(123)

# Dividend growth delta_t
phi_d, sig_d = 0.2, 0.02
delta_d = simulate_ar1(T, phi=phi_d, sigma=sig_d, x0=0.0, rng=rng)  # length T+1

# Risk aversion gamma_t starting at gamma_bar (gamma0 = gamma_bar)
gamma_bar = 2.0
phi_g, sig_g = 0.95, 0.10
gamma = simulate_ar1(T, phi=phi_g, sigma=sig_g, x0=gamma_bar, rng=rng)  # length T+1

params = dict(
    b_pd_d=0.5,   # dividend growth effect
    b_pd_g=-0.8,
    b_r_d=1.0,
    b_r_g=0.3,
)

pd_ratio, r = compute_pd_and_returns(delta_d=delta_d, gamma=gamma, params=params)

df = pd.DataFrame({"r": r, "pd": pd_ratio, "deld": delta_d, "gamma": gamma})

In [ ]:
burn = 1000
pd_g_component = params["b_pd_g"] * df["gamma"].iloc[burn:]
pd_d_component = params["b_pd_d"] * df["deld"].iloc[burn:]

print("std(gamma component):", np.std(pd_g_component))
print("std(delta_d component):", np.std(pd_d_component))
print("ratio std(gamma)/std(delta_d):", np.std(pd_g_component) / np.std(pd_d_component))

approx_err = df["pd"].iloc[burn:] - params["b_pd_g"] * df["gamma"].iloc[burn:]
print("std(pd - b_pd_g*gamma):", np.std(approx_err))
print("std(pd):", np.std(df["pd"].iloc[burn:]))

**Result:** the risk-aversion-driven component of the pd ratio has ~25x the standard
deviation of the dividend-growth-driven component in this calibration, and removing the
risk-aversion term from `pd` leaves almost no residual variance (`std(pd - b_pd_g*gamma) ≈
std(pd)`) — confirming, in simulation, that discount-rate (risk-aversion) shocks are doing
essentially all the work in moving the pd ratio, exactly as the analytical decomposition in
Part 2 predicted.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

n_plot = 200
axes[0].plot(df["deld"].iloc[burn:burn+n_plot]);  axes[0].set_title("Dividend growth (dd)")
axes[1].plot(df["gamma"].iloc[burn:burn+n_plot]);  axes[1].set_title("Risk aversion (y)")
axes[2].plot(df["pd"].iloc[burn:burn+n_plot]);     axes[2].set_title("P/D ratio (pd)")
axes[3].plot(df["r"].iloc[burn:burn+n_plot]);      axes[3].set_title("Market returns (r)")
axes[3].set_xlabel("t (after burn-in)")
plt.tight_layout()
plt.show()

# Co-movement, demeaned
idx = slice(burn, burn + n_plot)
g = df["gamma"].iloc[idx] - df["gamma"].iloc[idx].mean()
p = df["pd"].iloc[idx] - df["pd"].iloc[idx].mean()

plt.figure(figsize=(10, 4))
plt.plot(g, label="gamma (demeaned)")
plt.plot(p, label="pd (demeaned)")
plt.legend()
plt.title("Co-movement (demeaned, post burn-in)")
plt.show()

There is clear negative co-movement between risk aversion and the pd ratio: spikes in
$\gamma$ coincide with drops in $pd$, consistent with $\gamma$ acting as a time-varying
discount-rate state variable — when risk aversion spikes, valuations compress.

## Part 4: VAR(1) on returns, pd ratio, and dividend growth

Estimate $z_t = \phi_0 + \phi_1 z_{t-1} + \varepsilon_t$ for $z_t = [r_t, pd_t, \Delta d_t]'$,
check stationarity, and see which of the three series are predictable from their own and each
other's lags.

In [ ]:
# Estimate VAR(1) for z_t = [r_t, pd_t, dd_t]'
Z = df[["r", "pd", "deld"]].iloc[1:]  # t=1..T, so lag exists

res = VAR(Z).fit(1, trend="c")

phi0 = res.params.iloc[0].to_numpy()   # intercept vector
Phi1 = res.coefs[0]                    # lag-1 matrix

print("phi0:", phi0)
print("Phi1:\n", Phi1)

# R^2 per equation
E = res.resid
Y = res.model.endog[res.k_ar:]

# sum of squares by equation
Y_centered = Y - Y.mean(axis=0, keepdims=True)
TSS = np.sum(Y_centered**2, axis=0)

# Residual sum of squares by equation
RSS = np.sum(np.asarray(E)**2, axis=0)

R2 = 1 - RSS / TSS
r2_dict = dict(zip(res.names, R2))
print("R^2:", r2_dict)

# Stationarity check
eigvals = np.linalg.eigvals(Phi1)
print("Eigenvalues(Phi1):", eigvals)
print("max |eig|:", np.max(np.abs(eigvals)))
print("Stationary VAR?", "YES" if np.max(np.abs(eigvals)) < 1 else "NO")

**Result:** the fitted VAR(1) is stationary (max eigenvalue of the lag matrix ≈ 0.95, comfortably
below 1, as it must be for an economically sensible model built from stationary AR(1) primitives).
Both the pd ratio (R² ≈ 0.90) and, notably, **returns themselves (R² ≈ 0.87)** are highly predictable
from the lagged state. That is not an efficient-markets violation here — it is a direct consequence of
how the simulation is built: returns load contemporaneously on the same highly persistent
risk-aversion factor (φ = 0.95) that drives the pd ratio, so a persistent state variable
that is itself forecastable makes *everything* downstream of it forecastable too. Dividend growth,
by contrast, is only weakly predictable (R² ≈ 0.04), consistent with it being driven by its own
much-less-persistent AR(1) (φ = 0.2). The practical takeaway: return predictability in this class of
model is a symptom of discount-rate persistence, not of any mispricing — exactly the mechanism the
Part 2 variance decomposition points to.